# STT Benchmark: Audio Condition & Noise Reduction Impact

Sean Tallman  
AI Audio Evaluation Portfolio

## Goal

This project evaluates how audio conditions (noise, reverb, codec)
and preprocessing (noise reduction) impact speech-to-text accuracy.

We compare:
- Whisper
- Google STT
- AssemblyAI

Metrics:
- WER / CER
- S/D/I breakdown
- PESQ / STOI
- RTF

In [ ]:
import pandas as pd

real_df = pd.read_csv("../benchmark_output/openai_clean_results.csv")
len(real_df)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/test_transcripts.csv")

df.head()

In [ ]:
print(f"Total clips: {len(df)}")
print(df["condition"].value_counts())

In [ ]:
sample_results = [
    {
        "audio_file": "common_voice_en_30513358.mp3",
        "engine": "whisper",
        "condition": "clean",
        "wer": 0.08,
        "cer": 0.03,
        "rtf": 0.42
    },
    {
        "audio_file": "common_voice_en_43618790.mp3",
        "engine": "whisper",
        "condition": "clean",
        "wer": 0.04,
        "cer": 0.02,
        "rtf": 0.39
    },
    {
        "audio_file": "common_voice_en_21099981.mp3",
        "engine": "whisper",
        "condition": "clean",
        "wer": 0.11,
        "cer": 0.05,
        "rtf": 0.44
    }
]

In [ ]:
results_df = pd.DataFrame(sample_results)
results_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(results_df["audio_file"], results_df["wer"])
plt.ylabel("WER")
plt.xlabel("Audio File")
plt.title("Sample Whisper WER on Clean Clips")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
results_df.to_csv("../benchmark_output/first_sample_results.csv", index=False)
print("Saved to benchmark_output/first_sample_results.csv")

In [ ]:
import numpy as np

results_df = df.copy()

# simulate realistic values
np.random.seed(42)

results_df["engine"] = "whisper"
results_df["wer"] = np.round(np.random.uniform(0.03, 0.15, size=len(df)), 3)
results_df["cer"] = np.round(np.random.uniform(0.01, 0.06, size=len(df)), 3)
results_df["rtf"] = np.round(np.random.uniform(0.3, 0.6, size=len(df)), 2)

results_df

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["path"], results_df["wer"])
plt.ylabel("WER")
plt.xlabel("Audio File")
plt.title("Simulated Whisper WER on 12 Clean Clips")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
results_df.to_csv("../benchmark_output/first_sample_results.csv", index=False)

In [ ]:
import pandas as pd

real_df = pd.read_csv("../benchmark_output/openai_clean_results.csv")
real_df.head()

In [ ]:
import pandas as pd

real_df = pd.read_csv("../benchmark_output/openai_clean_results.csv")
print("row_count =", len(real_df))
real_df.head()

In [ ]:
from jiwer import wer

real_df["wer"] = real_df.apply(
    lambda row: wer(
        str(row["reference_text"]).lower(),
        str(row["hypothesis_text"]).lower()
    ),
    axis=1
)

real_df[["audio_file", "wer"]]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(real_df["audio_file"], real_df["wer"])
plt.ylabel("WER")
plt.xlabel("Audio File")
plt.title("OpenAI Transcription WER (12 Clean Clips)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
print("Average WER:", round(real_df["wer"].mean(), 4))

## 7. Export final results
Prepare and save clean benchmark output for portfolio use.

In [ ]:
final_df = df.copy()

In [ ]:
import pandas as pd

df = pd.read_csv("benchmark_output/openai_clean_results.csv")

df.head()

In [ ]:
import os
os.getcwd()

In [ ]:
os.listdir()

In [ ]:
os.path.exists("benchmark_output")

In [ ]:
os.path.exists("benchmark_output/openai_clean_results.csv")

In [ ]:
import os
os.getcwd()

In [ ]:
os.listdir("..")

In [ ]:
os.listdir("../..")

In [ ]:
os.listdir("/Users/seantallman/Library/Mobile Documents/com~apple~CloudDocs")

In [ ]:
import pandas as pd

df = pd.read_csv("../benchmark_output/openai_clean_results.csv")
df.head()

In [ ]:
import pandas as pd
import os

csv_path = "../benchmark_output/openai_clean_results.csv"

print("Notebook working dir:", os.getcwd())
print("CSV exists:", os.path.exists(csv_path))

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df)
else:
    print("CSV not found.")

In [ ]:
import pandas as pd
from jiwer import wer, process_words

df = pd.read_csv("../benchmark_output/openai_clean_results.csv")

scored_rows = []

for _, row in df.iterrows():
    ref = str(row["reference_text"])
    hyp = str(row["hypothesis_text"])

    measures = process_words(ref, hyp)

    scored_rows.append({
        "audio_file": row["audio_file"],
        "wer": measures.wer,
        "substitutions": measures.substitutions,
        "deletions": measures.deletions,
        "insertions": measures.insertions,
        "reference_word_count": len(ref.split()),
        "hypothesis_word_count": len(hyp.split())
    })

scored_df = pd.DataFrame(scored_rows)
scored_df

In [ ]:
print(scored_df.shape)
print(f"Average WER: {scored_df['wer'].mean():.4f}")

In [ ]:
final_df = scored_df.copy()

final_df = final_df[[
    "audio_file",
    "wer",
    "substitutions",
    "deletions",
    "insertions",
    "reference_word_count",
    "hypothesis_word_count"
]]

final_df = final_df.sort_values(by="wer", ascending=False).reset_index(drop=True)
final_df["wer"] = final_df["wer"].round(4)

avg_wer = final_df["wer"].mean()

summary_row = {
    "audio_file": "AVERAGE",
    "wer": round(avg_wer, 4),
    "substitutions": "",
    "deletions": "",
    "insertions": "",
    "reference_word_count": "",
    "hypothesis_word_count": ""
}

final_df = pd.concat([final_df, pd.DataFrame([summary_row])], ignore_index=True)

output_path = "../benchmark_output/openai_clean_final_results.csv"
final_df.to_csv(output_path, index=False)

print(f"Saved final results to: {output_path}")


In [ ]:
final_df = scored_df.copy()

final_df = final_df[[
    "audio_file",
    "wer",
    "substitutions",
    "deletions",
    "insertions",
    "reference_word_count",
    "hypothesis_word_count"
]]

final_df = final_df.sort_values(by="wer", ascending=False).reset_index(drop=True)
final_df["wer"] = final_df["wer"].round(4)

avg_wer = final_df["wer"].mean()

summary_row = {
    "audio_file": "AVERAGE",
    "wer": round(avg_wer, 4),
    "substitutions": "",
    "deletions": "",
    "insertions": "",
    "reference_word_count": "",
    "hypothesis_word_count": ""
}

import pandas as pd
final_df = pd.concat([final_df, pd.DataFrame([summary_row])], ignore_index=True)

output_path = "../benchmark_output/openai_clean_final_results.csv"
final_df.to_csv(output_path, index=False)

print(f"Saved final results to: {output_path}")

final_df

In [24]:
import pandas as pd
pd.read_csv("../benchmark_output/openai_clean_final_results.csv")

,audio_file,wer,substitutions,deletions,insertions,reference_word_count,hypothesis_word_count
0,common_voice_en_17922255.mp3,0.7500,3.0,0.0,0.0,4.0,4.0
1,common_voice_en_17877635.mp3,0.1667,1.0,0.0,1.0,12.0,13.0
2,common_voice_en_17257228.mp3,0.1111,2.0,0.0,0.0,18.0,18.0
3,common_voice_en_30513358.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
4,common_voice_en_18983699.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
5,common_voice_en_40145147.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
6,common_voice_en_20792443.mp3,0.0714,1.0,0.0,0.0,14.0,14.0
7,common_voice_en_43618790.mp3,0.0000,0.0,0.0,0.0,9.0,9.0
8,common_voice_en_21099981.mp3,0.0000,0.0,0.0,0.0,10.0,10.0
9,common_voice_en_36941027.mp3,0.0000,0.0,0.0,0.0,11.0,11.0


In [25]:
import pandas as pd

df = pd.read_csv("../benchmark_output/openai_clean_final_results.csv")
df

,audio_file,wer,substitutions,deletions,insertions,reference_word_count,hypothesis_word_count
0,common_voice_en_17922255.mp3,0.7500,3.0,0.0,0.0,4.0,4.0
1,common_voice_en_17877635.mp3,0.1667,1.0,0.0,1.0,12.0,13.0
2,common_voice_en_17257228.mp3,0.1111,2.0,0.0,0.0,18.0,18.0
3,common_voice_en_30513358.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
4,common_voice_en_18983699.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
5,common_voice_en_40145147.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
6,common_voice_en_20792443.mp3,0.0714,1.0,0.0,0.0,14.0,14.0
7,common_voice_en_43618790.mp3,0.0000,0.0,0.0,0.0,9.0,9.0
8,common_voice_en_21099981.mp3,0.0000,0.0,0.0,0.0,10.0,10.0
9,common_voice_en_36941027.mp3,0.0000,0.0,0.0,0.0,11.0,11.0


In [28]:
import pandas as pd

df = pd.read_csv("../benchmark_output/openai_clean_final_results.csv")
df

,audio_file,wer,substitutions,deletions,insertions,reference_word_count,hypothesis_word_count
0,common_voice_en_17922255.mp3,0.7500,3.0,0.0,0.0,4.0,4.0
1,common_voice_en_17877635.mp3,0.1667,1.0,0.0,1.0,12.0,13.0
2,common_voice_en_17257228.mp3,0.1111,2.0,0.0,0.0,18.0,18.0
3,common_voice_en_30513358.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
4,common_voice_en_18983699.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
5,common_voice_en_40145147.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
6,common_voice_en_20792443.mp3,0.0714,1.0,0.0,0.0,14.0,14.0
7,common_voice_en_43618790.mp3,0.0000,0.0,0.0,0.0,9.0,9.0
8,common_voice_en_21099981.mp3,0.0000,0.0,0.0,0.0,10.0,10.0
9,common_voice_en_36941027.mp3,0.0000,0.0,0.0,0.0,11.0,11.0


In [29]:
import pandas as pd

df = pd.read_csv("../benchmark_output/openai_clean_final_results.csv")

avg_wer = df["wer"].mean()

summary_row = {
    "audio_file": "AVERAGE",
    "wer": round(avg_wer, 4),
    "substitutions": "",
    "deletions": "",
    "insertions": "",
    "reference_word_count": "",
    "hypothesis_word_count": ""
}

df = pd.concat([df, pd.DataFrame([summary_row])], ignore_index=True)

df.to_csv("../benchmark_output/openai_clean_final_results.csv", index=False)

df

,audio_file,wer,substitutions,deletions,insertions,reference_word_count,hypothesis_word_count
0,common_voice_en_17922255.mp3,0.7500,3.0,0.0,0.0,4.0,4.0
1,common_voice_en_17877635.mp3,0.1667,1.0,0.0,1.0,12.0,13.0
2,common_voice_en_17257228.mp3,0.1111,2.0,0.0,0.0,18.0,18.0
3,common_voice_en_30513358.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
4,common_voice_en_18983699.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
5,common_voice_en_40145147.mp3,0.0833,1.0,0.0,0.0,12.0,12.0
6,common_voice_en_20792443.mp3,0.0714,1.0,0.0,0.0,14.0,14.0
7,common_voice_en_43618790.mp3,0.0000,0.0,0.0,0.0,9.0,9.0
8,common_voice_en_21099981.mp3,0.0000,0.0,0.0,0.0,10.0,10.0
9,common_voice_en_36941027.mp3,0.0000,0.0,0.0,0.0,11.0,11.0
